# 📁 Data Exploration



## Purpose

This notebook is the **entry point** for the entire project. It does two things:

1. **Extracts** the raw GRAZPEDWRI-DX dataset from the ZIP file into the `IronGear/dataset/` folder.
2. **Explores** the dataset so we fully understand its structure, class distribution, patient demographics, and annotation quality before any modelling work begins.

The outputs (a `dataset_paths.json` config and an `exploration_summary.json`) are saved to `IronGear/data/reports/`

---

## Dataset: GRAZPEDWRI-DX

- **Source:** https://www.kaggle.com/datasets/jasonroggy/grazpedwri-dx
- **Content:** Pediatric wrist trauma X-rays from the Medical University of Graz, Austria
- **Size:** 20,327 images from 6,091 unique patients
- **Format:** Images split across 4 folders, YOLO-format `.txt` annotation files
- **Classes:** 9 annotated finding classes (we use 5 in Sprint 1)
- **Metadata:** `dataset.csv` with patient_id, age, gender, laterality, projection, diagnosis

---

## Restart Safety

> ✅ **Every cell is self-contained.**
> After a kernel restart — run **Cell 2 (Paths)** first, then any cell independently.

---

## Execution Order

```
Cell 1  →  Install dependencies
Cell 2  →  Set up all paths           ⚠️ Always run after restart
Cell 3  →  Extract dataset ZIP         (skips if already done)
Cell 4  →  Locate key dataset paths    (saves dataset_paths.json)
Cell 5  →  Read class mapping from meta.yaml
Cell 6  →  Scan images, labels and metadata
Cell 7  →  Count annotations per class (reveals class imbalance)
Cell 8  →  Analyse patient demographics
Cell 9  →  Generate and save exploration figures
Cell 10 →  Display sample annotated images
Cell 11 →  Save exploration summary JSON
```


## 1 — Install Dependencies

In [1]:
import subprocess, sys

# Install packages quietly. check=True raises CalledProcessError if any install fails.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "matplotlib", "pandas", "pyyaml", "tqdm", "opencv-python"],
    check=True
)
print("✓ All dependencies ready.")

✓ All dependencies ready.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
amazon-sagemaker-jupyter-ai-q-developer 1.2.9 requires numpy<=2.0.1, but you have numpy 2.4.3 which is incompatible.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.3 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.3 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.3 which is incompatible.
autogluon-features 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have 

## 2 — Path Setup ⚠️
---

> **Always run this cell first after any kernel restart.**
> All other cells depend on the variables defined here.

### EFS Auto-Detection
SageMaker mounts EFS at different base paths depending on domain configuration. We check four common candidates in order and use the first one where the `IronGear` folder already exists.

### Directory Layout Created

```
IronGear/
├── dataset/              ← ZIP lives here; we also extract here
│   └── grazpedwri-dx.zip
└── data/                 ← all processed outputs (shared by Sprint 1, 2, 3)
    ├── reports/          ← JSON summaries and path config
    └── figures/
        └── exploration/  ← figures generated by this notebook
```


In [13]:
from pathlib import Path

# ─── EFS Mount Auto-Detection ─────────────────────────────────────────────────
# Check candidate paths in order of likelihood on SageMaker Studio.
# The first path where IronGear/ exists is used as the EFS base.
_CANDIDATES = [
    Path("/home/sagemaker-user/user-default-efs"),  # most common SageMaker mount
    Path("/home/sagemaker-user"),                   # fallback 1
    Path.home() / "user-default-efs",               # fallback 2
    Path.home(),                                     # last resort
]
EFS      = next((c for c in _CANDIDATES if (c / "IronGear").exists()), _CANDIDATES[0])
IRONGEAR = EFS / "IronGear"

# ─── Input Paths ──────────────────────────────────────────────────────────────
DATASET_ZIP = IRONGEAR / "dataset" / "grazpedwri-dx.zip"  # original Kaggle download
EXTRACT_DIR = IRONGEAR / "dataset"   # unzip to same folder so ZIP and contents are together

# ─── Output Paths (shared across all sprints) ─────────────────────────────────
DATA_DIR    = IRONGEAR / "data"                      # root for all processed data
FIG_DIR     = DATA_DIR / "figures" / "exploration"   # figures from this notebook
REPORTS_DIR = DATA_DIR / "reports"                   # JSON summaries and configs

# Create output directories (parents=True handles nested creation)
for d in [FIG_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Validate ZIP Exists ──────────────────────────────────────────────────────
assert DATASET_ZIP.exists(), (
    f"\n✗ Dataset ZIP not found: {DATASET_ZIP}"
    f"\n  Home dir     : {Path.home()}"
    f"\n  EFS detected : {EFS}"
    f"\n  Please confirm the ZIP is at IronGear/dataset/grazpedwri-dx.zip"
)

print("✓ Paths configured")
print(f"  EFS base     : {EFS}")
print(f"  IronGear     : {IRONGEAR}")
print(f"  Dataset ZIP  : {DATASET_ZIP}  ({DATASET_ZIP.stat().st_size / 1e9:.2f} GB)")
print(f"  Extract dir  : {EXTRACT_DIR}")
print(f"  Reports dir  : {REPORTS_DIR}")
print(f"  Figures dir  : {FIG_DIR}")

✓ Paths configured
  EFS base     : /home/sagemaker-user/user-default-efs
  IronGear     : /home/sagemaker-user/user-default-efs/IronGear
  Dataset ZIP  : /home/sagemaker-user/user-default-efs/IronGear/dataset/grazpedwri-dx.zip  (16.26 GB)
  Extract dir  : /home/sagemaker-user/user-default-efs/IronGear/dataset
  Reports dir  : /home/sagemaker-user/user-default-efs/IronGear/data/reports
  Figures dir  : /home/sagemaker-user/user-default-efs/IronGear/data/figures/exploration


## 3 — Extract Dataset

In [3]:
import zipfile

# ─── Marker-based skip logic ──────────────────────────────────────────────────
marker = EXTRACT_DIR / ".extracted"  # hidden file — presence = already extracted

if marker.exists():
    print("✓ Dataset already extracted — skipping.")
    print(f"  (Delete {marker} to force re-extraction)")
else:
    print(f"Extracting {DATASET_ZIP.name}...")
    print("  This may take 5–15 minutes depending on EFS speed.")

    with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
        print(f"  ZIP contains {len(zf.namelist()):,} files")
        zf.extractall(EXTRACT_DIR)   # extract all to EXTRACT_DIR

    marker.touch()   # create marker so we skip next time
    print("\n✓ Extraction complete.")

# ─── Display Top-Level Contents ───────────────────────────────────────────────
# Confirm structure looks correct after extraction
print("\nExtracted contents:")
for p in sorted(EXTRACT_DIR.iterdir()):
    if p.name.startswith("."):        # skip hidden files
        continue
    if p.is_dir():
        n_files = sum(1 for _ in p.rglob("*") if _.is_file())  # count all nested files
        print(f"  📁  {p.name:<35}  ({n_files:,} files)")
    else:
        print(f"  📄  {p.name:<35}  ({p.stat().st_size / 1e6:.1f} MB)")

Extracting grazpedwri-dx.zip...
  This may take 5–15 minutes depending on EFS speed.
  ZIP contains 81,314 files

✓ Extraction complete.

Extracted contents:
  📄  dataset.csv                          (1.9 MB)
  📁  folder_structure                     (60,986 files)
  📄  grazpedwri-dx.zip                    (16256.7 MB)
  📁  images_part1                         (5,031 files)
  📁  images_part2                         (5,143 files)
  📁  images_part3                         (4,842 files)
  📁  images_part4                         (5,311 files)


---
## 4 — Locate Key Dataset Paths

Dynamically finds the four critical paths inside the extracted dataset using `rglob` (recursive search). Using `rglob` rather than hardcoded paths makes this robust against ZIP archives that unpack into a nested subdirectory.


In [4]:
import json

# ─── Helper: find first file matching name anywhere under root ─────────────────
def find_first(root: Path, name: str) -> Path:
    """
    Recursively search `root` for a file named `name`.
    Returns the first match (sorted for determinism).
    Raises FileNotFoundError if nothing is found.
    """
    matches = sorted(root.rglob(name))
    if not matches:
        raise FileNotFoundError(
            f"'{name}' not found under {root}.\n"
            f"Ensure the dataset was extracted correctly."
        )
    return matches[0]

# ─── Locate Key Files ─────────────────────────────────────────────────────────
DATASET_CSV = find_first(EXTRACT_DIR, "dataset.csv")    # patient metadata CSV
META_YAML   = find_first(EXTRACT_DIR, "meta.yaml")       # class name mapping YAML
LABELS_DIR  = META_YAML.parent / "labels"                # YOLO labels dir (sibling of meta.yaml)

# Collect all images_part* folders (children of dataset.csv's parent directory)
IMAGE_DIRS = sorted([
    p for p in DATASET_CSV.parent.iterdir()
    if p.is_dir() and p.name.startswith("images_part")
])

# ─── Validate ─────────────────────────────────────────────────────────────────
assert LABELS_DIR.exists(), (
    f"Labels dir not found: {LABELS_DIR}. "
    "Expected a 'labels/' folder beside meta.yaml in the yolov5 directory."
)
assert len(IMAGE_DIRS) > 0, (
    f"No 'images_part*' folders found under {DATASET_CSV.parent}."
)

# ─── Save Path Config for Downstream Notebooks ────────────────────────────────
path_config = {
    "dataset_csv" : str(DATASET_CSV),
    "meta_yaml"   : str(META_YAML),
    "labels_dir"  : str(LABELS_DIR),
    "image_dirs"  : [str(d) for d in IMAGE_DIRS],
    "extract_dir" : str(EXTRACT_DIR),
    "data_dir"    : str(DATA_DIR),
}
config_path = REPORTS_DIR / "dataset_paths.json"
with open(config_path, "w") as f:
    json.dump(path_config, f, indent=2)

print("✓ All dataset paths located and validated")
print(f"\n  dataset.csv  : {DATASET_CSV}")
print(f"  meta.yaml    : {META_YAML}")
print(f"  labels dir   : {LABELS_DIR}")
print(f"  image folders: {[d.name for d in IMAGE_DIRS]}")
print(f"\n✓ Path config saved → {config_path}")
print("  (Loaded automatically by Notebook data_processing )")

✓ All dataset paths located and validated

  dataset.csv  : /home/sagemaker-user/user-default-efs/IronGear/dataset/dataset.csv
  meta.yaml    : /home/sagemaker-user/user-default-efs/IronGear/dataset/folder_structure/yolov5/meta.yaml
  labels dir   : /home/sagemaker-user/user-default-efs/IronGear/dataset/folder_structure/yolov5/labels
  image folders: ['images_part1', 'images_part2', 'images_part3', 'images_part4']

✓ Path config saved → /home/sagemaker-user/user-default-efs/IronGear/data/reports/dataset_paths.json
  (Loaded automatically by Notebook data_processing )


---
## 5 — Read Class Mapping from meta.yaml

`meta.yaml` maps integer class IDs (0–8) to human-readable finding names. We read it here and plan which classes belong to each sprint.


> **Note on raw names:** The meta.yaml uses compact names like `periostealreaction` (no underscore). In Notebook data_processing we map these to cleaner project names like `periosteal_reaction`.


In [6]:
import yaml

# ─── Load meta.yaml ───────────────────────────────────────────────────────────
with open(META_YAML) as f:
    meta = yaml.safe_load(f)

names = meta.get("names", {})

# Handle both list format (index = class ID) and dict format (key = class ID)
if isinstance(names, list):
    RAW_CLASSES = {i: n for i, n in enumerate(names)}     # list → {0: name, 1: name, ...}
elif isinstance(names, dict):
    RAW_CLASSES = {int(k): v for k, v in names.items()}   # dict → {0: name, 1: name, ...}
else:
    raise ValueError(f"Unexpected format for 'names' in meta.yaml: {type(names)}")

# ─── Sprint Classification ─────────────────────────────────────────────────────
# These are the EXACT lowercase raw names from meta.yaml
SPRINT1_TARGETS    = {"fracture", "metal", "periostealreaction", "pronatorsign", "softtissue"}
SPRINT2_ADDITIONAL = {"boneanomaly", "bonelesion", "foreignbody"}
# 'text' is a radiologist annotation overlay — not a clinical finding, so ignored

# ─── Mapping of raw names → clean project names (for display) ─────────────────
RAW_TO_PROJECT_DISPLAY = {
    "fracture"          : "fracture",
    "metal"             : "metal_implant",
    "periostealreaction": "periosteal_reaction",
    "pronatorsign"      : "pronator_sign",
    "softtissue"        : "soft_tissue",
    "boneanomaly"       : "bone_anomaly",
    "bonelesion"        : "bone_lesion",
    "foreignbody"       : "foreign_body",
}

# ─── Display Table ────────────────────────────────────────────────────────────
print(f"Found {len(RAW_CLASSES)} raw classes in meta.yaml:")
print()
print(f"  {'ID':>4}  {'Raw Name in meta.yaml':<25}  {'Project Name':<28}  Sprint")
print("  " + "─" * 75)

for cid, cname in sorted(RAW_CLASSES.items()):
    cname_lower = cname.lower()
    proj        = RAW_TO_PROJECT_DISPLAY.get(cname_lower, "—")
    sprint_tag  = (
        "✓  Sprint 1" if cname_lower in SPRINT1_TARGETS else
        "→  Sprint 2" if cname_lower in SPRINT2_ADDITIONAL else
        "   (ignored)"
    )
    print(f"  {cid:>4}  {cname:<25}  {proj:<28}  {sprint_tag}")

print()
print("Legend:")
print("  ✓ Sprint 1  — used immediately (5 classes, this sprint)")
print("  → Sprint 2  — added in Sprint 2 for 8-class detection")
print("     Ignored  — radiologist text overlay, not a clinical finding")

Found 9 raw classes in meta.yaml:

    ID  Raw Name in meta.yaml      Project Name                  Sprint
  ───────────────────────────────────────────────────────────────────────────
     0  boneanomaly                bone_anomaly                  →  Sprint 2
     1  bonelesion                 bone_lesion                   →  Sprint 2
     2  foreignbody                foreign_body                  →  Sprint 2
     3  fracture                   fracture                      ✓  Sprint 1
     4  metal                      metal_implant                 ✓  Sprint 1
     5  periostealreaction         periosteal_reaction           ✓  Sprint 1
     6  pronatorsign               pronator_sign                 ✓  Sprint 1
     7  softtissue                 soft_tissue                   ✓  Sprint 1
     8  text                       —                                (ignored)

Legend:
  ✓ Sprint 1  — used immediately (5 classes, this sprint)
  → Sprint 2  — added in Sprint 2 for 8-class detectio

---
## 6 — Scan Images, Labels and Metadata

Performs a full file system scan to count all images, labels and metadata rows.

Every image has a corresponding `.txt` label file — even images with no findings have an empty `.txt` file. These empty files represent **negative samples** (healthy wrists) which are essential for training a detector that doesn't produce constant false positives.


In [7]:
import pandas as pd
from tqdm.notebook import tqdm
from collections import Counter

# Accepted image file extensions
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# ─── Scan All Image Folders ───────────────────────────────────────────────────
# Collect every image across all 4 image_part folders into a flat list of records
img_records = []
print("Scanning image folders:")

for folder in IMAGE_DIRS:
    folder_count = 0
    for p in sorted(folder.rglob("*")):           # recursive scan
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            img_records.append({
                "stem"   : p.stem,                # bare filename without extension
                "path"   : str(p),                # full absolute path
                "ext"    : p.suffix.lower(),      # lowercase extension
                "folder" : folder.name,            # which images_part folder
                "size_kb": round(p.stat().st_size / 1024, 1),  # size in KB
            })
            folder_count += 1
    print(f"  {folder.name:<20}: {folder_count:,} images")

# Drop duplicates (same stem in multiple folders — defensive, shouldn't happen)
img_df = pd.DataFrame(img_records).drop_duplicates("stem", keep="first")
print(f"\n  Total unique images : {len(img_df):,}")
print(f"  Avg image size      : {img_df['size_kb'].mean():.0f} KB")
print(f"  File formats        : {img_df['ext'].value_counts().to_dict()}")

# ─── Scan Label Folder ────────────────────────────────────────────────────────
# Each .txt file uses the same stem as its corresponding image.
# Empty .txt = no findings in that image (negative sample).
lbl_records = []
for p in sorted(LABELS_DIR.rglob("*.txt")):
    if p.is_file():
        lbl_records.append({"stem": p.stem, "label_path": str(p)})

lbl_df = pd.DataFrame(lbl_records).drop_duplicates("stem", keep="first")

# Count empty vs non-empty label files
n_empty_labels = sum(1 for _, r in lbl_df.iterrows()
                     if Path(r["label_path"]).stat().st_size == 0)
print(f"\n  Total label files   : {len(lbl_df):,}")
print(f"  Empty (no findings) : {n_empty_labels:,}   ← negative samples")
print(f"  Non-empty           : {len(lbl_df) - n_empty_labels:,}   ← annotated images")

# ─── Load dataset.csv (Patient Metadata) ──────────────────────────────────────
meta_df = pd.read_csv(DATASET_CSV)
print(f"\n  Metadata rows       : {len(meta_df):,}")
print(f"  Columns available   : {list(meta_df.columns)}")

Scanning image folders:
  images_part1        : 5,031 images
  images_part2        : 5,143 images
  images_part3        : 4,842 images
  images_part4        : 5,311 images

  Total unique images : 20,327
  Avg image size      : 780 KB
  File formats        : {'.png': 20327}

  Total label files   : 20,327
  Empty (no findings) : 15   ← negative samples
  Non-empty           : 20,312   ← annotated images

  Metadata rows       : 20,327
  Columns available   : ['filestem', 'patient_id', 'study_number', 'timehash', 'gender', 'age', 'laterality', 'projection', 'initial_exam', 'ao_classification', 'cast', 'diagnosis_uncertain', 'osteopenia', 'fracture_visible', 'metal', 'pixel_spacing', 'device_manufacturer']


---
## 7 — Count Annotations per Class

Reads every label file and counts how many bounding box annotations exist per raw class. This reveals the **class imbalance** — one of the central challenges of this project.

> ⏱️ **Expected runtime:** 2–5 minutes (reads ~20k text files)


In [8]:
# ─── Annotation Counting ──────────────────────────────────────────────────────
# For each label file, we parse every YOLO-format line and:
#   1. Increment class_counts[class_id] for each bounding box annotation
#   2. Increment image_cls_count[class_id] once per image (not per annotation)
#   3. Track n_empty for images with no annotations

class_counts    = Counter()   # {class_id: total_bounding_box_count}
image_cls_count = Counter()   # {class_id: number_of_images_containing_class}
n_empty         = 0           # number of negative-sample images
n_total_annots  = 0           # grand total of all bounding boxes

print("Counting annotations across all label files...")

for _, row in tqdm(lbl_df.iterrows(), total=len(lbl_df), desc="Reading labels"):
    classes_this_image = set()   # track unique classes per image for image_cls_count

    with open(row["label_path"]) as f:
        lines = [l.strip() for l in f if l.strip()]   # skip blank lines

    if not lines:
        n_empty += 1    # empty file = image has no annotated findings
        continue

    for line in lines:
        parts = line.split()
        if len(parts) == 5:              # valid YOLO line: class_id cx cy w h
            cid = int(parts[0])          # class ID is always the first column
            class_counts[cid] += 1
            classes_this_image.add(cid)
            n_total_annots += 1

    for cid in classes_this_image:       # increment image count once per class per image
        image_cls_count[cid] += 1

# ─── Display Results Table ────────────────────────────────────────────────────
print(f"\n  Total annotations      : {n_total_annots:,}")
print(f"  Images with no findings: {n_empty:,}  (negative samples)")
print()
print(f"  {'ID':>4}  {'Raw Class':<25}  {'Annotations':>12}  {'% total':>8}  {'Images':>8}  {'Sprint'}")
print("  " + "─" * 80)

for cid, cname in sorted(RAW_CLASSES.items()):
    annots  = class_counts.get(cid, 0)
    imgs    = image_cls_count.get(cid, 0)
    pct     = annots / n_total_annots * 100 if n_total_annots > 0 else 0
    sprint  = (
        "Sprint 1" if cname.lower() in SPRINT1_TARGETS else
        "Sprint 2" if cname.lower() in SPRINT2_ADDITIONAL else
        "ignored"
    )
    print(f"  {cid:>4}  {cname:<25}  {annots:>12,}  {pct:>7.1f}%  {imgs:>8,}  {sprint}")

print()
print("⚠️  Class imbalance: fracture is ~40x more common than soft_tissue.")
print("   This will be fixed in Notebook 01 via oversampling of rare classes.")

Counting annotations across all label files...


Reading labels:   0%|          | 0/20327 [00:00<?, ?it/s]


  Total annotations      : 47,443
  Images with no findings: 15  (negative samples)

    ID  Raw Class                   Annotations   % total    Images  Sprint
  ────────────────────────────────────────────────────────────────────────────────
     0  boneanomaly                         276      0.6%       192  Sprint 2
     1  bonelesion                           45      0.1%        42  Sprint 2
     2  foreignbody                           8      0.0%         8  Sprint 2
     3  fracture                         18,090     38.1%    13,550  Sprint 1
     4  metal                               818      1.7%       707  Sprint 1
     5  periostealreaction                3,453      7.3%     2,235  Sprint 1
     6  pronatorsign                        567      1.2%       566  Sprint 1
     7  softtissue                          464      1.0%       439  Sprint 1
     8  text                             23,722     50.0%    20,274  ignored

⚠️  Class imbalance: fracture is ~40x more common tha

---
## 8 — Patient Demographics Analysis

Analyses the patient-level metadata from `dataset.csv`. Understanding the patient distribution is critical for designing the patient-level train/val/test split in Notebook 01.

**Rule:** A patient ID must appear in exactly one of train, val, or test — never across multiple sets.


In [9]:
# ─── Detect Metadata Columns ──────────────────────────────────────────────────
# Normalise column names to lowercase for robust matching
cols_lower = {c.lower().strip(): c for c in meta_df.columns}

# Find patient_id column (try multiple common naming conventions)
patient_col = (
    cols_lower.get("patient_id") or
    cols_lower.get("patientid") or
    cols_lower.get("patient")
)
assert patient_col, f"patient_id column not found. Available: {list(meta_df.columns)}"

# ─── Patient-Level Statistics ─────────────────────────────────────────────────
n_patients  = meta_df[patient_col].nunique()              # count unique patients
imgs_per_pt = meta_df.groupby(patient_col).size()         # images per patient Series

print("Patient Statistics")
print("─" * 48)
print(f"  Unique patients            : {n_patients:,}")
print(f"  Total images               : {len(meta_df):,}")
print(f"  Avg images per patient     : {imgs_per_pt.mean():.1f}")
print(f"  Median images per patient  : {imgs_per_pt.median():.1f}")
print(f"  Max images per patient     : {imgs_per_pt.max():,}")
print(f"  Min images per patient     : {imgs_per_pt.min():,}")
print(f"  Patients with 1 image      : {(imgs_per_pt == 1).sum():,}")
print(f"  Patients with 2–5 images   : {((imgs_per_pt >= 2) & (imgs_per_pt <= 5)).sum():,}")
print(f"  Patients with >5 images    : {(imgs_per_pt > 5).sum():,}")

# ─── Age Statistics ───────────────────────────────────────────────────────────
if "age" in cols_lower:
    age_col = cols_lower["age"]
    age_series = meta_df[age_col].dropna()
    print(f"\nAge Statistics")
    print("─" * 48)
    print(f"  Min    : {age_series.min():.1f} years")
    print(f"  Max    : {age_series.max():.1f} years")
    print(f"  Mean   : {age_series.mean():.1f} years")
    print(f"  Std    : {age_series.std():.1f} years")
    print(f"  → Confirms this is a paediatric (0–18 year) dataset")

# ─── Gender Distribution ──────────────────────────────────────────────────────
if "gender" in cols_lower:
    gc = cols_lower["gender"]
    gc_counts = meta_df[gc].value_counts()
    print(f"\nGender Distribution  (column: '{gc}')")
    print("─" * 48)
    for gender, count in gc_counts.items():
        print(f"  {str(gender):<6}: {count:>6,}  ({count/len(meta_df)*100:.1f}%)")

# ─── Laterality Distribution ──────────────────────────────────────────────────
if "laterality" in cols_lower:
    lc = cols_lower["laterality"]
    lat_counts = meta_df[lc].value_counts()
    print(f"\nLaterality (L/R wrist)  (column: '{lc}')")
    print("─" * 48)
    for side, count in lat_counts.items():
        print(f"  {str(side):<6}: {count:>6,}  ({count/len(meta_df)*100:.1f}%)")

Patient Statistics
────────────────────────────────────────────────
  Unique patients            : 6,091
  Total images               : 20,327
  Avg images per patient     : 3.3
  Median images per patient  : 2.0
  Max images per patient     : 30
  Min images per patient     : 1
  Patients with 1 image      : 96
  Patients with 2–5 images   : 4,892
  Patients with >5 images    : 1,103

Age Statistics
────────────────────────────────────────────────
  Min    : 0.2 years
  Max    : 19.0 years
  Mean   : 10.9 years
  Std    : 3.6 years
  → Confirms this is a paediatric (0–18 year) dataset

Gender Distribution  (column: 'gender')
────────────────────────────────────────────────
  M     : 12,040  (59.2%)
  F     :  8,285  (40.8%)
  O     :      2  (0.0%)

Laterality (L/R wrist)  (column: 'laterality')
────────────────────────────────────────────────
  L     : 11,135  (54.8%)
  R     :  9,192  (45.2%)


---
## Cell 9 — Exploration Figures

Generates a 2×3 grid of plots providing a visual overview of the dataset. All figures are saved to `IronGear/data/figures/exploration/` and displayed inline.



In [10]:
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — safe for SageMaker
import matplotlib.pyplot as plt
import numpy as np

# Colour palette — one colour per raw class
COLORS_9 = ["#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6",
            "#1abc9c","#e67e22","#34495e","#e91e63"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("GRAZPEDWRI-DX — Dataset Exploration Overview",
             fontsize=15, fontweight="bold", y=1.01)

# ─── Plot 1: Annotation count per raw class ───────────────────────────────────
ax = axes[0, 0]
sorted_ids   = sorted(class_counts.keys())
class_names  = [RAW_CLASSES[i] for i in sorted_ids]
annot_vals   = [class_counts[i] for i in sorted_ids]
bar_colors   = [COLORS_9[i % 9] for i in range(len(class_names))]
bars = ax.barh(class_names, annot_vals, color=bar_colors, edgecolor="white")
ax.set_title("Annotations per Raw Class", fontweight="bold")
ax.set_xlabel("Number of Annotations")
ax.invert_yaxis()   # largest at top
for bar, v in zip(bars, annot_vals):
    ax.text(v + 30, bar.get_y() + bar.get_height() / 2,
            f"{v:,}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.3)

# ─── Plot 2: Images containing each class ─────────────────────────────────────
ax = axes[0, 1]
img_vals = [image_cls_count[i] for i in sorted_ids]
ax.barh(class_names, img_vals, color=bar_colors, edgecolor="white")
ax.set_title("Images Containing Each Class", fontweight="bold")
ax.set_xlabel("Number of Images")
ax.invert_yaxis()
for i, v in enumerate(img_vals):
    ax.text(v + 10, i, f"{v:,}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.3)

# ─── Plot 3: Images per folder ────────────────────────────────────────────────
ax = axes[0, 2]
folder_counts = img_df["folder"].value_counts().sort_index()
ax.bar(range(len(folder_counts)), folder_counts.values,
       color="#3498db", edgecolor="white")
ax.set_xticks(range(len(folder_counts)))
ax.set_xticklabels(folder_counts.index, rotation=20, ha="right")
ax.set_title("Images per Folder", fontweight="bold")
ax.set_ylabel("Image Count")
for i, v in enumerate(folder_counts.values):
    ax.text(i, v + 20, f"{v:,}", ha="center", fontsize=9)
ax.grid(axis="y", alpha=0.3)

# ─── Plot 4: Images per patient histogram ─────────────────────────────────────
ax = axes[1, 0]
ax.hist(imgs_per_pt.values, bins=30, color="#2ecc71", edgecolor="white", alpha=0.85)
ax.axvline(imgs_per_pt.mean(), color="red", linestyle="--",
           label=f"Mean = {imgs_per_pt.mean():.1f}")
ax.axvline(imgs_per_pt.median(), color="orange", linestyle=":",
           label=f"Median = {imgs_per_pt.median():.1f}")
ax.set_title("Images per Patient Distribution", fontweight="bold")
ax.set_xlabel("Images per Patient")
ax.set_ylabel("Patient Count")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

# ─── Plot 5: Annotation coverage by sprint ────────────────────────────────────
ax = axes[1, 1]
s1_ann  = sum(class_counts[i] for i,n in RAW_CLASSES.items() if n.lower() in SPRINT1_TARGETS)
s2_ann  = sum(class_counts[i] for i,n in RAW_CLASSES.items() if n.lower() in SPRINT2_ADDITIONAL)
oth_ann = n_total_annots - s1_ann - s2_ann
ax.pie(
    [s1_ann, s2_ann, oth_ann],
    labels=[f"Sprint 1 (5 cls)\n{s1_ann:,}",
            f"Sprint 2 (3 more)\n{s2_ann:,}",
            f"Ignored\n{oth_ann:,}"],
    colors=["#2ecc71", "#3498db", "#bdc3c7"],
    autopct="%1.1f%%", startangle=90,
    textprops={"fontsize": 8},
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
ax.set_title("Annotation Coverage by Sprint", fontweight="bold")

# ─── Plot 6: Age distribution ─────────────────────────────────────────────────
ax = axes[1, 2]
if "age" in cols_lower:
    age_col = cols_lower["age"]
    ax.hist(meta_df[age_col].dropna(), bins=40, color="#9b59b6",
            edgecolor="white", alpha=0.85)
    ax.set_title("Patient Age Distribution", fontweight="bold")
    ax.set_xlabel("Age (years)")
    ax.set_ylabel("Number of Patients")
    ax.grid(axis="y", alpha=0.3)
else:
    ax.text(0.5, 0.5, "Age data unavailable",
            ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()

# Save to figures directory and display inline
fig_path = FIG_DIR / "01_dataset_overview.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Figure saved: {fig_path}")

✓ Figure saved: /home/sagemaker-user/user-default-efs/IronGear/data/figures/exploration/01_dataset_overview.png


---
## 10 — Sample Annotated Images

Displays 2 sample images per raw class with ground-truth bounding boxes rendered. This provides a qualitative understanding of what each finding looks like on a paediatric wrist X-ray.

### Visual Key
- Each class has a unique colour for its bounding boxes
- Class name is printed above each box
- Images may contain multiple classes (overlapping boxes)


In [11]:
import cv2
import matplotlib.patches as mpatches

# Colour map for bounding boxes — normalised (R, G, B) values for matplotlib
CLASS_COLORS_MPL = {
    0: (0.90, 0.20, 0.20),   # red
    1: (0.20, 0.55, 0.90),   # blue
    2: (0.20, 0.80, 0.30),   # green
    3: (0.90, 0.65, 0.10),   # gold
    4: (0.65, 0.20, 0.90),   # purple
    5: (0.10, 0.80, 0.80),   # cyan
    6: (0.95, 0.45, 0.10),   # orange
    7: (0.60, 0.85, 0.20),   # lime
    8: (0.85, 0.20, 0.60),   # pink
}

# Fast lookup: image stem → full image path
stem_to_img = dict(zip(img_df["stem"], img_df["path"]))

# Select up to 2 images per class (only images whose file exists on disk)
samples_to_show = []   # list of (stem, label_path, main_class_id)

for cid in sorted(RAW_CLASSES.keys()):
    found = 0
    for _, row in lbl_df.iterrows():
        if found >= 2:
            break
        with open(row["label_path"]) as f:
            lines = [l.strip() for l in f if l.strip()]
        cls_ids = [int(l.split()[0]) for l in lines if len(l.split()) == 5]

        # Include this image only if it contains the target class and the image file exists
        if cid in cls_ids and row["stem"] in stem_to_img:
            samples_to_show.append((row["stem"], row["label_path"], cid))
            found += 1

samples_to_show = samples_to_show[:12]   # 12 samples = clean 3×4 grid

# ─── Render 3×4 Grid ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle(
    "Sample Annotated Images — All 9 Raw Classes\n"
    "(Coloured boxes = YOLO ground truth annotations)",
    fontsize=13, fontweight="bold"
)

for idx, (stem, lbl_path, main_cls) in enumerate(samples_to_show):
    ax = axes[idx // 4][idx % 4]
    ax.axis("off")

    # Load image: OpenCV reads as BGR, matplotlib expects RGB — convert
    img_bgr = cv2.imread(stem_to_img[stem])
    if img_bgr is None:
        ax.text(0.5, 0.5, "Image not found",
                ha="center", va="center", transform=ax.transAxes)
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W    = img_rgb.shape[:2]   # pixel dimensions for coordinate conversion

    ax.imshow(img_rgb, cmap="gray")

    # Parse label file and draw each bounding box
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue   # skip malformed or empty lines

            # YOLO format: class_id  cx  cy  bw  bh  (all normalised 0–1)
            cid_box        = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])

            # Convert normalised YOLO coords to pixel coordinates for matplotlib
            x1 = (cx - bw / 2) * W    # top-left x pixel
            y1 = (cy - bh / 2) * H    # top-left y pixel

            color = CLASS_COLORS_MPL.get(cid_box, (0.7, 0.7, 0.7))
            ax.add_patch(mpatches.Rectangle(
                (x1, y1), bw * W, bh * H,
                linewidth=2, edgecolor=color, facecolor="none"
            ))
            ax.text(x1 + 2, y1 - 4,
                    RAW_CLASSES.get(cid_box, str(cid_box)),
                    color=color, fontsize=7.5, fontweight="bold",
                    bbox=dict(facecolor="black", alpha=0.45,
                              pad=1, edgecolor="none"))

    ax.set_title(f"{RAW_CLASSES.get(main_cls, '?')}\n{stem[:22]}",
                 fontsize=8, fontweight="bold")

# Hide unused subplot panels
for idx in range(len(samples_to_show), 12):
    axes[idx // 4][idx % 4].set_visible(False)

plt.tight_layout()
fig_path2 = FIG_DIR / "02_sample_annotations.png"
plt.savefig(fig_path2, dpi=120, bbox_inches="tight")
plt.show()
print(f"✓ Figure saved: {fig_path2}")

✓ Figure saved: /home/sagemaker-user/user-default-efs/IronGear/data/figures/exploration/02_sample_annotations.png


---
## 11 — Save Exploration Summary

Saves all exploration findings to `IronGear/data/reports/exploration_summary.json`. This JSON file acts as a record of the dataset characteristics for project reporting and is also checked by Notebook data_processing to confirm the dataset was correctly explored.

### Contents Saved
- Dataset identity and Kaggle URL
- Total image, label and annotation counts
- Per-class annotation and image counts
- Patient statistics (count, avg images per patient)
- Sprint-to-class allocation plan
- Timestamp


In [12]:
import datetime

# ─── Compile Summary Dictionary ───────────────────────────────────────────────
# All numeric values are cast to Python native int/float
# because numpy int64/float64 are not JSON-serialisable by default
exploration_summary = {
    # Provenance
    "generated_at" : datetime.datetime.now().isoformat(),
    "dataset"      : "GRAZPEDWRI-DX",
    "source_url"   : "https://www.kaggle.com/datasets/jasonroggy/grazpedwri-dx",

    # Image and label counts
    "total_images"        : int(len(img_df)),
    "total_label_files"   : int(len(lbl_df)),
    "empty_label_files"   : int(n_empty),
    "total_annotations"   : int(n_total_annots),
    "image_folders"       : [d.name for d in IMAGE_DIRS],

    # Patient information
    "unique_patients"          : int(n_patients),
    "avg_images_per_patient"   : round(float(imgs_per_pt.mean()), 2),
    "median_images_per_patient": round(float(imgs_per_pt.median()), 2),
    "max_images_per_patient"   : int(imgs_per_pt.max()),

    # Class information
    "raw_classes"           : {str(k): v for k, v in RAW_CLASSES.items()},
    "annotations_per_class" : {RAW_CLASSES[k]: int(v) for k, v in class_counts.items()},
    "images_per_class"      : {RAW_CLASSES[k]: int(v) for k, v in image_cls_count.items()},

    # Sprint planning
    "sprint1_classes"    : sorted(SPRINT1_TARGETS),
    "sprint2_additional" : sorted(SPRINT2_ADDITIONAL),
}

# Save to disk
summary_path = REPORTS_DIR / "exploration_summary.json"
with open(summary_path, "w") as f:
    json.dump(exploration_summary, f, indent=2)

# ─── Final Summary Output ─────────────────────────────────────────────────────
print("=" * 60)
print(" NOTEBOOK 00 COMPLETE — Data Exploration")
print("=" * 60)
print(f"\n  Dataset          : GRAZPEDWRI-DX")
print(f"  Total images     : {exploration_summary['total_images']:,}")
print(f"  Unique patients  : {exploration_summary['unique_patients']:,}")
print(f"  Total annotations: {exploration_summary['total_annotations']:,}")
print(f"  Negative images  : {exploration_summary['empty_label_files']:,}")
print()
print("  Files saved:")
print(f"    ✓ {REPORTS_DIR / 'dataset_paths.json'}")
print(f"    ✓ {summary_path}")
print(f"    ✓ {FIG_DIR / '01_dataset_overview.png'}")
print(f"    ✓ {FIG_DIR / '02_sample_annotations.png'}")
print()
print("  Next step  →  Run: IronGear/data_processing.ipynb")

 NOTEBOOK 00 COMPLETE — Data Exploration

  Dataset          : GRAZPEDWRI-DX
  Total images     : 20,327
  Unique patients  : 6,091
  Total annotations: 47,443
  Negative images  : 15

  Files saved:
    ✓ /home/sagemaker-user/user-default-efs/IronGear/data/reports/dataset_paths.json
    ✓ /home/sagemaker-user/user-default-efs/IronGear/data/reports/exploration_summary.json
    ✓ /home/sagemaker-user/user-default-efs/IronGear/data/figures/exploration/01_dataset_overview.png
    ✓ /home/sagemaker-user/user-default-efs/IronGear/data/figures/exploration/02_sample_annotations.png

  Next step  →  Run: IronGear/data_processing.ipynb
